# 03 — Mamba SSM v4: TBM-Native Selection + Boruta + Engineering Fixes

**Task ID:** `MODEL-MAMBA-ZERO-FEE-008`
**Version:** 0.4
**Date:** 2026-06-01

## What changed vs v3

| # | v3 | v4 |
|---|----|----|
| 1 | Selection label: 1h return proxy | Selection label: TBM (actual training target) |
| 2 | Stage 2: MI ranking → top 60 | Stage 2: Univariate AUC filter against TBM |
| 3 | Stage 3: MI stability (3 sub-windows) | Stage 3: Rolling AUC stability per feature (AUC > 0.51 in ≥ 2/3 windows) |
| 4 | Stage 4: LGBM permutation → top 20 (arbitrary K) | Stage 4: Boruta-LGBM (shadow features, no arbitrary K) |
| 5 | 2 subsets × 2 targets = 4 runs (M2 and M4 differed by 1 feature) | 1 subset × 1 target = 1 run |
| 6 | On-the-fly sequence slicing in DataLoader | Pre-built `(N, seq_len, F)` array in RAM |
| 7 | d_state=16, batch=256 | d_state=8, batch=512 |

## Pipeline
```
V1 (195) + V4 (25) = 220 features
        │
        ▼ Stage 1: Variance drop + Spearman collinearity (ρ > 0.85)
        ▼ Stage 2: Univariate AUC filter against TBM label
        ▼ Stage 3: Rolling AUC stability (3 windows, AUC > 0.51 in ≥ 2)
        ▼ Stage 4: Boruta-LGBM (shadow features — no fixed K)
        │
        ▼
  Mamba SSM (d_model=64, 2 layers, d_state=8, chunk=8)
  1 run: all Boruta-selected features vs label_tbm
```

In [1]:
import json
import math
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.0)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Device: Apple MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device: CUDA ({torch.cuda.get_device_name()})')
else:
    DEVICE = torch.device('cpu')
    print('Device: CPU')

Device: Apple MPS


/Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys

def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO     = _find_repo_root()
RAW_DIR  = REPO / 'data' / 'raw'
FEAT_DIR = REPO / 'data' / 'features'
ARTS_DIR = REPO / 'artifacts' / '03_mamba_omni_0fee_v4'
ARTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / 'src'))

# ── Sequence ──────────────────────────────────────────────────────────────────
SYMBOL     = 'BTCUSDT'
INTERVAL   = '1h'
SEQ_LEN    = 24
STRIDE     = 1
BATCH      = 512      # v4: 512 (fewer MPS dispatches vs v3's 256)
EPOCHS     = 60
LR         = 3e-4
FOCAL_GAMMA = 2.0

# ── Mamba ─────────────────────────────────────────────────────────────────────
D_MODEL    = 64
D_STATE    = 8        # v4: 8 (halved vs v3's 16 — major scan speedup)
D_CONV     = 4
EXPAND     = 2
N_LAYERS   = 2
SCAN_CHUNK = 8        # L/chunk = 24/8 = 3 Python carries per forward pass
DROPOUT    = 0.1

# ── Feature selection ─────────────────────────────────────────────────────────
STAGE2_AUC_FLOOR    = 0.505   # univariate AUC must beat this (either direction)
STAGE3_N_WINDOWS    = 3
STAGE3_MIN_WINS     = 2
STAGE3_AUC_THRESH   = 0.510
BORUTA_TRIALS       = 30      # shadow-feature comparison rounds
BORUTA_ESTIMATORS   = 150     # trees per LGBM trial
BORUTA_HIT_RATE     = 0.5     # feature must beat shadow in > 50% of trials

# ── Splits ────────────────────────────────────────────────────────────────────
OOS_START = pd.Timestamp('2024-01-01')
TRAIN_END = pd.Timestamp('2022-12-31')
VAL_END   = pd.Timestamp('2023-12-31')

# TBM label params (same as v2/v3)
SL_MULT  = 1.5
TP_MULT  = 1.5
MAX_HOLD = SEQ_LEN   # 24 bars

SELECTION_CACHE = ARTS_DIR / 'selected_features.json'

print(f'Repo: {REPO}')
print(f'SEQ_LEN={SEQ_LEN}  BATCH={BATCH}  D_STATE={D_STATE}  SCAN_CHUNK={SCAN_CHUNK}')
print(f'BORUTA: {BORUTA_TRIALS} trials × {BORUTA_ESTIMATORS} trees')

Repo: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
SEQ_LEN=24  BATCH=512  D_STATE=8  SCAN_CHUNK=8
BORUTA: 30 trials × 150 trees


---
## Phase A — Load V1 + V4 Features

In [3]:
v4_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet')
v4_df.index = v4_df.index.tz_localize(None) if v4_df.index.tz else v4_df.index
V4_FEATURE_COLS = list(v4_df.columns)
print(f'V4: {len(V4_FEATURE_COLS)} features')

v1_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
v1_df.index = v1_df.index.tz_localize(None) if v1_df.index.tz else v1_df.index

with open(FEAT_DIR / 'feature_registry.json') as f:
    registry = json.load(f)
BACKTEST_COLS = registry.get('backtest_only_cols', [])

_EXCLUDE = set(BACKTEST_COLS) | {
    'open', 'high', 'low', 'close', 'volume',
    'label', 'tbm_label', 'fh_label', 'forward_ret',
}
V1_FEATURE_COLS = [
    c for c in v1_df.columns
    if c not in _EXCLUDE and pd.api.types.is_numeric_dtype(v1_df[c])
]
print(f'V1: {len(V1_FEATURE_COLS)} features')

raw_df = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet',
                          columns=['open', 'high', 'low', 'close', 'volume'])
raw_df.index = raw_df.index.tz_localize(None) if raw_df.index.tz else raw_df.index

merged = v1_df.copy()
for col in V4_FEATURE_COLS:
    merged[col] = v4_df[col].reindex(merged.index)
merged['high']  = raw_df['high'].reindex(merged.index)
merged['low']   = raw_df['low'].reindex(merged.index)
merged['close'] = raw_df['close'].reindex(merged.index)
merged.index    = merged.index.tz_localize(None) if merged.index.tz else merged.index

ALL_FEATURE_COLS = V1_FEATURE_COLS + V4_FEATURE_COLS
print(f'Merged: {len(merged):,} rows  |  {len(ALL_FEATURE_COLS)} features')
print(f'Range:  {merged.index.min().date()} → {merged.index.max().date()}')

V4: 25 features
V1: 195 features
Merged: 74,366 rows  |  220 features
Range:  2017-11-15 → 2026-05-16


---
## Phase B — TBM-Native Feature Selection (4 Stages)

**Key difference from v2/v3:** All stages use the **TBM label** as target — not a 1h return proxy.  
This means every filter asks: *"does this feature predict whether TP or SL is hit within 24h?"*

| Stage | Method | Removes |
|-------|--------|---------|
| 1 | Variance drop + Spearman collinearity ρ > 0.85 | Constant & duplicate features |
| 2 | Univariate AUC filter vs TBM | Features with zero directional signal |
| 3 | Rolling AUC stability (3 windows) | Features predictive in one regime only |
| 4 | Boruta-LGBM shadow test | Features that can't beat random noise consistently |

In [4]:
# ── Compute TBM label for the train-selection window ─────────────────────────
# Done here (Phase B) so all selection stages use the actual training target.

close_s = raw_df['close'].reindex(merged.index)
close_s.index = close_s.index.tz_localize(None) if close_s.index.tz else close_s.index
hi_s = raw_df['high'].reindex(merged.index); hi_s.index = close_s.index
lo_s = raw_df['low'].reindex(merged.index);  lo_s.index = close_s.index

tr_    = pd.concat([(hi_s - lo_s),
                    (hi_s - close_s.shift(1)).abs(),
                    (lo_s - close_s.shift(1)).abs()], axis=1).max(axis=1)
atr_s  = tr_.ewm(span=14, adjust=False).mean()

close_arr = close_s.values
atr_arr   = atr_s.values
n_total   = len(close_arr)
y_tbm_arr = np.full(n_total, np.nan)

print(f'Computing TBM labels for full dataset (n={n_total:,}) ...')
t0 = time.time()
for i in range(n_total - MAX_HOLD):
    if np.isnan(atr_arr[i]) or atr_arr[i] == 0:
        continue
    entry = close_arr[i]
    tp, sl = entry + TP_MULT * atr_arr[i], entry - SL_MULT * atr_arr[i]
    for j in range(i + 1, min(i + MAX_HOLD + 1, n_total)):
        if close_arr[j] >= tp: y_tbm_arr[i] = 1; break
        if close_arr[j] <= sl: y_tbm_arr[i] = 0; break

y_tbm_full = pd.Series(y_tbm_arr, index=merged.index, name='label_tbm')
print(f'Done in {time.time()-t0:.0f}s  |  positive rate: {np.nanmean(y_tbm_arr):.3f}  '
      f'|  valid: {np.isfinite(y_tbm_arr).sum():,} / {n_total:,}')

# ── Selection window: pre-OOS train period ────────────────────────────────────
sel_mask   = (merged.index < OOS_START) & y_tbm_full.notna()
df_sel     = merged[sel_mask][ALL_FEATURE_COLS].fillna(0)
y_sel      = y_tbm_full[sel_mask].values.astype(int)
X_sel_raw  = df_sel.values.astype(np.float32)
print(f'Selection window: {sel_mask.sum():,} valid TBM bars × {X_sel_raw.shape[1]} features')

Computing TBM labels for full dataset (n=74,366) ...
Done in 0s  |  positive rate: 0.511  |  valid: 63,216 / 74,366
Selection window: 44,962 valid TBM bars × 220 features


In [5]:
# ── Stage 1: Variance drop + Spearman collinearity ──────────────────────────
print('Stage 1: Variance drop + Spearman collinearity ...')
t0 = time.time()

# Variance
var_mask = X_sel_raw.var(axis=0) > 1e-6
cols1 = [c for c, v in zip(ALL_FEATURE_COLS, var_mask) if v]
X1    = X_sel_raw[:, var_mask]
print(f'  After variance drop: {X_sel_raw.shape[1]} → {len(cols1)}')

# Spearman collinearity
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    corr_mat, _ = spearmanr(X1)
corr_mat = np.abs(np.array(corr_mat))
np.fill_diagonal(corr_mat, 0.0)

removed = set()
for i in range(len(cols1)):
    if i in removed: continue
    for j in range(i + 1, len(cols1)):
        if j not in removed and corr_mat[i, j] > 0.85:
            removed.add(j)

cols1 = [c for i, c in enumerate(cols1) if i not in removed]
X1    = X1[:, [i for i in range(X1.shape[1]) if i not in removed]]
print(f'  After collinearity (ρ>0.85): → {len(cols1)} survivors')
print(f'  Stage 1 done in {time.time()-t0:.1f}s')

Stage 1: Variance drop + Spearman collinearity ...
  After variance drop: 220 → 219
  After collinearity (ρ>0.85): → 125 survivors
  Stage 1 done in 0.5s


In [6]:
# ── Stage 2: Univariate AUC filter against TBM label ────────────────────────
# For each feature, compute roc_auc_score(y_tbm, feature_values).
# AUC < 0.5 means negative predictive power — flip to max(auc, 1-auc).
# Keep features with effective AUC > STAGE2_AUC_FLOOR.
print(f'Stage 2: Univariate AUC filter (threshold={STAGE2_AUC_FLOOR}) ...')
t0 = time.time()

aucs = []
for j in range(X1.shape[1]):
    try:
        a = roc_auc_score(y_sel, X1[:, j])
        aucs.append(max(a, 1 - a))
    except Exception:
        aucs.append(0.5)
aucs = np.array(aucs)

keep2  = aucs >= STAGE2_AUC_FLOOR
cols2  = [c for c, k in zip(cols1, keep2) if k]
X2     = X1[:, keep2]

print(f'  {len(cols1)} → {len(cols2)} survivors (AUC ≥ {STAGE2_AUC_FLOOR})')
print(f'  AUC distribution: min={aucs.min():.4f}  mean={aucs.mean():.4f}  '
      f'max={aucs.max():.4f}  p75={np.percentile(aucs,75):.4f}')
print(f'  Stage 2 done in {time.time()-t0:.1f}s')

Stage 2: Univariate AUC filter (threshold=0.505) ...
  125 → 67 survivors (AUC ≥ 0.505)
  AUC distribution: min=0.5001  mean=0.5073  max=0.5299  p75=0.5100
  Stage 2 done in 0.5s


In [7]:
# ── Stage 3: Rolling AUC stability across 3 windows ─────────────────────────
# A feature is "stable" if AUC > STAGE3_AUC_THRESH in ≥ STAGE3_MIN_WINS windows.
# Each window is a non-overlapping slice of the selection period.
# This kills features that are only predictive in one market regime.
print(f'Stage 3: Rolling AUC stability '
      f'({STAGE3_N_WINDOWS} windows, AUC>{STAGE3_AUC_THRESH} in ≥{STAGE3_MIN_WINS}) ...')
t0  = time.time()
n   = len(X2)
win = n // STAGE3_N_WINDOWS
hit_counts3 = np.zeros(len(cols2), dtype=int)

for w in range(STAGE3_N_WINDOWS):
    sl = slice(w * win, (w + 1) * win if w < STAGE3_N_WINDOWS - 1 else n)
    Xw, yw = X2[sl], y_sel[sl]
    if len(np.unique(yw)) < 2:
        continue
    for j in range(Xw.shape[1]):
        try:
            a = roc_auc_score(yw, Xw[:, j])
            if max(a, 1 - a) >= STAGE3_AUC_THRESH:
                hit_counts3[j] += 1
        except Exception:
            pass
    print(f'  Window {w+1}/{STAGE3_N_WINDOWS}: '
          f'{sl.start if sl.start else 0}–{n if sl.stop is None else sl.stop} bars  '
          f'({(hit_counts3 > 0).sum()} features with ≥1 hit so far)')

stable3 = hit_counts3 >= STAGE3_MIN_WINS
cols3   = [c for c, s in zip(cols2, stable3) if s]
X3      = X2[:, stable3]
print(f'  {len(cols2)} → {len(cols3)} stable survivors')
print(f'  Stage 3 done in {time.time()-t0:.1f}s')

Stage 3: Rolling AUC stability (3 windows, AUC>0.51 in ≥2) ...
  Window 1/3: 0–14987 bars  (47 features with ≥1 hit so far)
  Window 2/3: 14987–29974 bars  (59 features with ≥1 hit so far)
  Window 3/3: 29974–44962 bars  (65 features with ≥1 hit so far)
  67 → 41 stable survivors
  Stage 3 done in 0.3s


In [8]:
# ── Stage 4: Boruta-LGBM shadow feature test ─────────────────────────────────
# Each trial:
#   1. Create shadow X = column-wise permuted copy of X3
#   2. Train LGBM on [X3 | shadow_X]
#   3. Real feature "wins" if its importance > max(shadow importances)
# Keep features winning > 50% of trials (BORUTA_HIT_RATE).
# No need to guess K — Boruta sets its own cutoff from the data.

if SELECTION_CACHE.exists():
    with open(SELECTION_CACHE) as f:
        _cached = json.load(f)
    SELECTED_FEATURES = _cached['features']
    print(f'Loaded {len(SELECTED_FEATURES)} features from cache.')
else:
    print(f'Stage 4: Boruta ({BORUTA_TRIALS} trials × {BORUTA_ESTIMATORS} trees) ...')
    t0  = time.time()
    rng = np.random.default_rng(42)
    n_f = len(cols3)
    hits = np.zeros(n_f)

    for trial in tqdm(range(BORUTA_TRIALS), desc='  Boruta trials'):
        # Shadow features: permute each column independently
        X_shadow = X3.copy()
        for j in range(n_f):
            rng.shuffle(X_shadow[:, j])
        X_aug = np.hstack([X3, X_shadow])

        model = lgb.LGBMClassifier(
            n_estimators=BORUTA_ESTIMATORS, num_leaves=31, learning_rate=0.05,
            subsample=0.7, colsample_bytree=0.7, min_child_samples=20,
            verbose=-1, n_jobs=-1, random_state=int(trial),
        )
        model.fit(X_aug, y_sel)

        imp        = model.feature_importances_
        real_imp   = imp[:n_f]
        shadow_imp = imp[n_f:]
        threshold  = shadow_imp.max()
        hits      += (real_imp > threshold).astype(float)

    hit_rate = hits / BORUTA_TRIALS
    selected_mask  = hit_rate > BORUTA_HIT_RATE
    SELECTED_FEATURES = [c for c, s in zip(cols3, selected_mask) if s]

    elapsed = time.time() - t0
    print(f'  Boruta done in {elapsed:.0f}s')
    print(f'  {len(cols3)} → {len(SELECTED_FEATURES)} confirmed features')
    print(f'  Hit rates:')
    for feat, rate, sel in sorted(zip(cols3, hit_rate, selected_mask),
                                   key=lambda t: t[1], reverse=True):
        mark = '✓' if sel else '✗'
        print(f'    {mark} {feat:<40s}  {rate:.2f}')

    with open(SELECTION_CACHE, 'w') as f:
        json.dump({
            'features':    SELECTED_FEATURES,
            'hit_rates':   {c: float(r) for c, r in zip(cols3, hit_rate)},
            'stage_sizes': [len(ALL_FEATURE_COLS), len(cols1),
                            len(cols2), len(cols3), len(SELECTED_FEATURES)],
        }, f, indent=2)

print(f'\nFinal feature set ({len(SELECTED_FEATURES)}):')
for i, feat in enumerate(SELECTED_FEATURES, 1):
    tag = ' [V4]' if feat in V4_FEATURE_COLS else ''
    print(f'  {i:2d}. {feat}{tag}')

Loaded 39 features from cache.

Final feature set (39):
   1. ret_48h
   2. ret_72h
   3. ret_168h
   4. close_vs_ema_50
   5. macd_hist_12_26
   6. obv_z_72
   7. candles_since_cross
   8. ma_ribbon_width
   9. cloud_bullish
  10. supertrend_dist_15
  11. supertrend_dir_30
  12. fib_nearest_dist_48h
  13. fib_dist_618_48h
  14. fib_nearest_dist_168h
  15. close_vs_sma_504
  16. close_vs_sma_2160
  17. weekly_mom_accel
  18. halving_cycle_cos
  19. halving_cycle_pos
  20. dom_sin
  21. quarter_cos
  22. obv_divergence
  23. mfi_14
  24. cmf_20
  25. ad_z_168h
  26. dist_round_1000
  27. dist_round_10000
  28. atr_14_pct_rank
  29. bb_squeeze_20
  30. bb_squeeze_50
  31. hurst_168h
  32. kurt_24h
  33. skew_168h
  34. kurt_168h
  35. var_ratio_6h
  36. var_ratio_24h
  37. avg_trade_size [V4]
  38. adf_tstat_336h [V4]
  39. bb_width_pct [V4]


---
## Phase C — Splits & TBM Label

In [9]:
df_base = merged[ALL_FEATURE_COLS].copy()
df_base['label_tbm'] = y_tbm_full.reindex(df_base.index)
df_base['close']     = close_s.reindex(df_base.index)
df_base = df_base.dropna(subset=['close'])
df_base.index = df_base.index.tz_localize(None) if df_base.index.tz else df_base.index

train_mask = df_base.index <= TRAIN_END
val_mask   = (df_base.index > TRAIN_END) & (df_base.index <= VAL_END)
oos_mask   = df_base.index > VAL_END

df_tr  = df_base[train_mask].copy()
df_vl  = df_base[val_mask].copy()
df_oos = df_base[oos_mask].copy()

print(f'Train: {len(df_tr):,}  ({df_tr.index.min().date()} → {df_tr.index.max().date()})')
print(f'Val:   {len(df_vl):,}  ({df_vl.index.min().date()} → {df_vl.index.max().date()})')
print(f'OOS:   {len(df_oos):,}  ({df_oos.index.min().date()} → {df_oos.index.max().date()})')

# TBM label stats
for name, df in [('Train', df_tr), ('Val', df_vl)]:
    valid = df['label_tbm'].notna()
    pos   = df.loc[valid, 'label_tbm'].mean()
    print(f'{name} TBM: {valid.sum():,} valid ({valid.mean():.1%} of rows)  '
          f'positive rate={pos:.3f}')

Train: 44,799  (2017-11-15 → 2022-12-31)
Val:   8,759  (2022-12-31 → 2023-12-31)
OOS:   20,808  (2023-12-31 → 2026-05-16)
Train TBM: 37,751 valid (84.3% of rows)  positive rate=0.505
Val TBM: 7,195 valid (82.1% of rows)  positive rate=0.539


---
## Phase D — Normalization & Pre-Built Sequence Arrays

**v4 fix:** sequences are pre-allocated as `(N, seq_len, F)` float32 arrays in RAM before training.  
DataLoader indexes into a `TensorDataset` — no per-sample slicing, no `nan_to_num` in the hot path.

In [10]:
def build_sequences(X_scaled: np.ndarray, y: np.ndarray,
                    seq_len: int, stride: int = 1):
    """Pre-build (N_valid, seq_len, F) and (N_valid,) arrays."""
    ends     = np.arange(seq_len - 1, len(X_scaled), stride)
    valid    = ~np.isnan(y[ends].astype(float))
    ends     = ends[valid]
    X_seq    = np.stack([X_scaled[e - seq_len + 1 : e + 1] for e in ends], axis=0)
    y_seq    = y[ends].astype(np.int64)
    return X_seq.astype(np.float32), y_seq


def make_loaders(feat_cols, target_col='label_tbm',
                 batch=BATCH, seq_len=SEQ_LEN, stride=STRIDE):
    qt   = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
    X_tr_raw = np.nan_to_num(df_tr[feat_cols].values.astype(np.float32), nan=0.0)
    X_vl_raw = np.nan_to_num(df_vl[feat_cols].values.astype(np.float32), nan=0.0)
    X_tr = qt.fit_transform(X_tr_raw)
    X_vl = qt.transform(X_vl_raw)
    y_tr = df_tr[target_col].values
    y_vl = df_vl[target_col].values

    print(f'  Building training sequences (stride={stride}) ...', end=' ', flush=True)
    t0 = time.time()
    X_tr_seq, y_tr_seq = build_sequences(X_tr, y_tr, seq_len, stride)
    print(f'{len(y_tr_seq):,} seqs  ({time.time()-t0:.1f}s)')

    print(f'  Building validation sequences ...', end=' ', flush=True)
    t0 = time.time()
    X_vl_seq, y_vl_seq = build_sequences(X_vl, y_vl, seq_len, stride=1)
    print(f'{len(y_vl_seq):,} seqs  ({time.time()-t0:.1f}s)')

    mem_mb = (X_tr_seq.nbytes + X_vl_seq.nbytes) / 1e6
    print(f'  RAM for sequences: {mem_mb:.0f} MB')

    ds_tr = TensorDataset(torch.from_numpy(X_tr_seq), torch.from_numpy(y_tr_seq))
    ds_vl = TensorDataset(torch.from_numpy(X_vl_seq), torch.from_numpy(y_vl_seq))
    loader_tr = DataLoader(ds_tr, batch_size=batch, shuffle=True,  drop_last=True)
    loader_vl = DataLoader(ds_vl, batch_size=batch, shuffle=False, drop_last=False)
    return loader_tr, loader_vl, qt, len(feat_cols)

print('build_sequences + make_loaders ready.')

build_sequences + make_loaders ready.


---
## Phase E — Mamba Architecture (d_state=8, chunked scan)

`d_state` halved from v3's 16 → 8.  Scan tensors scale as `(B, n_chunks, chunk, D_inner, N)` —  
halving N halves every intermediate allocation in the scan.

In [11]:
def _chunked_ssm_scan(x, delta, A, B_mat, C_mat, D_coef, chunk=SCAN_CHUNK):
    B, L, D = x.shape
    N = A.shape[-1]
    log_dA = delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)
    dBu    = delta.unsqueeze(-1) * B_mat.unsqueeze(2) * x.unsqueeze(-1)
    n_chunks = (L + chunk - 1) // chunk
    L_pad    = n_chunks * chunk
    if L_pad > L:
        pad    = L_pad - L
        log_dA = F.pad(log_dA, (0, 0, 0, 0, 0, pad))
        dBu    = F.pad(dBu,    (0, 0, 0, 0, 0, pad))
        C_pad  = F.pad(C_mat,  (0, 0, 0, pad))
    else:
        C_pad = C_mat
    log_dA_c = log_dA.view(B, n_chunks, chunk, D, N)
    dBu_c    = dBu.view(B, n_chunks, chunk, D, N)
    C_c      = C_pad.view(B, n_chunks, chunk, N)
    log_P_c       = torch.cumsum(log_dA_c, dim=2)
    P_c           = torch.exp(log_P_c)
    within_c      = P_c * torch.cumsum(dBu_c / P_c.clamp(min=1e-12), dim=2)
    P_full_c      = torch.exp(log_dA_c.sum(dim=2))
    y = x.new_zeros(B, L_pad, D)
    h = x.new_zeros(B, D, N)
    for ci in range(n_chunks):
        h_chunk = P_c[:, ci] * h.unsqueeze(1) + within_c[:, ci]
        y[:, ci*chunk:(ci+1)*chunk] = (h_chunk * C_c[:, ci].unsqueeze(-2)).sum(-1)
        h = P_full_c[:, ci] * h + within_c[:, ci, -1]
    return y[:, :L] + x * D_coef.unsqueeze(0).unsqueeze(0)


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND):
        super().__init__()
        d_inner      = d_model * expand
        dt_rank      = max(1, d_model // 16)
        self.d_inner = d_inner; self.d_state = d_state; self.dt_rank = dt_rank
        A_init       = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log   = nn.Parameter(torch.log(A_init))
        self.D       = nn.Parameter(torch.ones(d_inner))
        self.norm    = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d  = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv - 1,
                                  groups=d_inner, bias=True)
        self.x_proj  = nn.Linear(d_inner, dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        nn.init.constant_(self.dt_proj.bias, math.log(math.expm1(1.0)))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x):
        res  = x; x = self.norm(x); L = x.shape[1]
        x_in, z = self.in_proj(x).chunk(2, dim=-1)
        x_in = F.silu(self.conv1d(x_in.transpose(1,2))[..., :L].transpose(1,2))
        dBC  = self.x_proj(x_in)
        dt, B_ssm, C = dBC.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dt))
        A     = -torch.exp(self.A_log.float())
        y     = _chunked_ssm_scan(x_in, delta, A, B_ssm, C, self.D)
        return self.out_proj(y * F.silu(z)) + res


class MambaClassifier(nn.Module):
    def __init__(self, n_features, d_model=D_MODEL, n_layers=N_LAYERS,
                 d_state=D_STATE, n_classes=3, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.blocks     = nn.ModuleList([MambaBlock(d_model, d_state) for _ in range(n_layers)])
        self.dropout    = nn.Dropout(dropout)
        self.head       = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = self.input_proj(x)
        for blk in self.blocks: x = blk(x)
        return self.head(self.dropout(x[:, -1, :]))


# Benchmark
_m = MambaClassifier(n_features=20).to(DEVICE)
_x = torch.randn(BATCH, SEQ_LEN, 20).to(DEVICE)
_m(_x)  # warm-up
t0 = time.perf_counter()
for _ in range(20): _m(_x)
ms = (time.perf_counter() - t0) / 20 * 1000
n_p = sum(p.numel() for p in _m.parameters())
print(f'MambaClassifier: {n_p:,} params  |  d_state={D_STATE}  batch={BATCH}')
print(f'Forward {BATCH}×{SEQ_LEN}×20 on {DEVICE}: {ms:.1f} ms/batch')
print(f'Est. fwd+bwd: {ms*3:.0f} ms  |  {len(range(1,EPOCHS+1))} epochs × ~? batches')
del _m, _x

MambaClassifier: 60,866 params  |  d_state=8  batch=512
Forward 512×24×20 on mps: 14.0 ms/batch
Est. fwd+bwd: 42 ms  |  60 epochs × ~? batches


---
## Phase F — Training Utilities

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('alpha', alpha)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def make_focal_loss(y_arr, gamma=FOCAL_GAMMA, device=DEVICE):
    counts = np.bincount(y_arr[~np.isnan(y_arr)].astype(int), minlength=3)
    alpha  = torch.tensor(1.0 / (counts + 1), dtype=torch.float32).to(device)
    return FocalLoss(gamma=gamma, alpha=alpha / alpha.sum())


def train_one_epoch(model, loader, criterion, opt, device):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    all_p, all_y = [], []
    for xb, yb in loader:
        p = torch.softmax(model(xb.to(device)), dim=-1)[:, 1].cpu().numpy()
        all_p.append(p); all_y.append(yb.numpy())
    p = np.concatenate(all_p); y = np.concatenate(all_y)
    # TBM has 3 classes (0=Down,1=Up,2=Flat); treat class-1 (Up) as positive
    y_bin = (y == 1).astype(int)
    auc = roc_auc_score(y_bin, p) if y_bin.sum() > 0 and y_bin.sum() < len(y_bin) else 0.5
    return p, y, auc

print('Utilities ready.')

Utilities ready.


: 

---
## Phase G — Training (1 Run: Boruta Features × TBM Label)

Single model. Full tqdm instrumentation with per-epoch metrics and ETA.

In [ ]:
TARGET_COL = 'label_tbm'
RUN_ID     = f'mamba_v4_boruta__{TARGET_COL}'
CACHE_PROBS = ARTS_DIR / f'probs_{RUN_ID}.npy'
CACHE_META  = ARTS_DIR / f'meta_{RUN_ID}.json'
RESULT      = None

if CACHE_PROBS.exists() and CACHE_META.exists():
    print(f'Loading from cache: {RUN_ID}')
    val_probs = np.load(CACHE_PROBS)
    with open(CACHE_META) as f:
        RESULT = json.load(f)
    RESULT['val_probs'] = val_probs
    print(f'  AUC={RESULT["best_auc"]:.4f}  ep={RESULT["best_epoch"]}')
else:
    print(f'Building loaders for {len(SELECTED_FEATURES)} features × {TARGET_COL} ...')
    tr_loader, vl_loader, qt, n_feat = make_loaders(SELECTED_FEATURES, TARGET_COL)
    print(f'  Batches/epoch: {len(tr_loader)}  |  val batches: {len(vl_loader)}')

    model    = MambaClassifier(n_features=n_feat).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')

    y_tr_vals = df_tr[TARGET_COL].dropna().values.astype(int)
    criterion = make_focal_loss(y_tr_vals)
    opt       = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched     = SequentialLR(opt,
                    [LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=5),
                     CosineAnnealingLR(opt, T_max=EPOCHS - 5, eta_min=1e-6)],
                    milestones=[5])

    # ── MPS warmup ────────────────────────────────────────────────────────────
    # Phase E benchmark used n_features=20; actual model has n_features=n_feat.
    # Different input shape → MPS compiles new Metal shaders on first real batch.
    # Without warmup this compilation blocks the first epoch for 30-60 minutes.
    # Warmup here forces compilation before the tqdm bar starts so all epochs
    # run at steady-state speed (~12 ms/batch forward on M-series).
    if DEVICE.type == 'mps':
        print(f'  MPS warmup ({BATCH}×{SEQ_LEN}×{n_feat}) — compiling shaders ...', end=' ', flush=True)
        t_wu = time.time()
        _xw  = torch.randn(BATCH, SEQ_LEN, n_feat, device=DEVICE)
        _yw  = torch.zeros(BATCH, dtype=torch.long, device=DEVICE)
        _lw  = criterion(model(_xw), _yw)   # fwd — compiles all forward kernels
        _lw.backward()                       # bwd — compiles all grad kernels
        opt.zero_grad()
        with torch.no_grad():
            model(_xw)                       # eval path
        torch.mps.synchronize()
        del _xw, _yw, _lw
        print(f'done in {time.time()-t_wu:.1f}s')

    best_auc = best_epoch = 0
    patience  = 12
    val_aucs, ep_times = [], []
    t_run = time.time()

    ep_bar = tqdm(
        range(1, EPOCHS + 1), desc=f'Training {RUN_ID}', unit='ep',
        bar_format='{l_bar}{bar}| ep {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]',
    )

    for epoch in ep_bar:
        t_ep    = time.time()
        tr_loss = train_one_epoch(model, tr_loader, criterion, opt, DEVICE)
        vl_probs, vl_y, vl_auc = eval_epoch(model, vl_loader, DEVICE)
        sched.step()

        ep_s = time.time() - t_ep
        ep_times.append(ep_s)
        val_aucs.append(vl_auc)

        if vl_auc > best_auc:
            best_auc = vl_auc; best_epoch = epoch
            best_probs = vl_probs.copy()
            torch.save(model.state_dict(), ARTS_DIR / f'model_{RUN_ID}.pt')

        eta_s = np.mean(ep_times[-5:]) * (EPOCHS - epoch)
        ep_bar.set_postfix({
            'loss': f'{tr_loss:.4f}',
            'val':  f'{vl_auc:.4f}',
            'best': f'{best_auc:.4f}@{best_epoch}',
            's/ep': f'{ep_s:.1f}',
            'ETA':  f'{eta_s/60:.1f}m',
        })

        if epoch - best_epoch >= patience:
            ep_bar.write(f'Early stop @ ep {epoch}  (best={best_auc:.4f} @ ep {best_epoch})')
            break

    ep_bar.close()
    elapsed = time.time() - t_run
    print(f'\nDone: AUC={best_auc:.4f}  best_ep={best_epoch}  '
          f'total={elapsed:.0f}s ({elapsed/60:.1f} min)  '
          f'avg {np.mean(ep_times):.1f}s/ep')

    np.save(CACHE_PROBS, best_probs)
    RESULT = {
        'run_id': RUN_ID, 'target': TARGET_COL,
        'best_auc': float(best_auc), 'best_epoch': int(best_epoch),
        'val_aucs': val_aucs,
        'n_features': int(n_feat), 'feat_cols': SELECTED_FEATURES,
        'n_train_seqs': len(tr_loader.dataset),
        'n_val_seqs':   len(vl_loader.dataset),
        'elapsed_s': float(elapsed),
        'ep_times': ep_times,
    }
    RESULT['val_probs'] = best_probs
    with open(CACHE_META, 'w') as f:
        json.dump({k: v for k, v in RESULT.items() if k != 'val_probs'}, f, indent=2)

Building loaders for 39 features × label_tbm ...
  Building training sequences (stride=1) ... 

37,728 seqs  (0.0s)
  Building validation sequences ... 7,179 seqs  (0.0s)
  RAM for sequences: 168 MB
  Batches/epoch: 73  |  val batches: 15
  Parameters: 62,082
  MPS warmup (512×24×39) — compiling shaders ... done in 0.2s


Training mamba_v4_boruta__label_tbm:   0%|          | ep 0/60 [00:00<?, ?ep/s]

---
## Phase H — Feature Attribution (Permutation Importance)

In [ ]:
CACHE_PERM = ARTS_DIR / f'perm_imp_{RUN_ID}.json'
PERM_IMP   = {}

if CACHE_PERM.exists():
    with open(CACHE_PERM) as f:
        PERM_IMP = json.load(f)
    print(f'Loaded from cache. Top 5: {sorted(PERM_IMP, key=PERM_IMP.get, reverse=True)[:5]}')
elif (ARTS_DIR / f'model_{RUN_ID}.pt').exists() and RESULT is not None:
    feat_cols  = RESULT['feat_cols']
    n_feat     = RESULT['n_features']
    base_auc   = RESULT['best_auc']

    _, vl_loader, _, _ = make_loaders(feat_cols, TARGET_COL, stride=1)
    model = MambaClassifier(n_features=n_feat).to(DEVICE)
    model.load_state_dict(torch.load(ARTS_DIR / f'model_{RUN_ID}.pt', map_location=DEVICE))
    model.eval()

    all_x, all_y = [], []
    with torch.no_grad():
        for xb, yb in vl_loader:
            all_x.append(xb); all_y.append(yb)
    X_clean = torch.cat(all_x, dim=0).to(DEVICE)
    Y_true  = torch.cat(all_y, dim=0).cpu().numpy()

    for fname in tqdm(feat_cols, desc='Permutation importance', unit='feat'):
        f_idx = feat_cols.index(fname)
        drops = []
        for _ in range(3):
            Xp = X_clean.clone()
            Xp[:, :, f_idx] = Xp[torch.randperm(Xp.shape[0]), :, f_idx]
            preds = torch.cat([
                torch.softmax(model(chunk), dim=-1)[:, 1].cpu()
                for chunk in torch.split(Xp, 512, dim=0)
            ]).numpy()
            drops.append(base_auc - roc_auc_score(Y_true, preds))
        PERM_IMP[fname] = float(np.mean(drops))

    with open(CACHE_PERM, 'w') as f:
        json.dump(PERM_IMP, f, indent=2)
    print(f'Top 5: {sorted(PERM_IMP, key=PERM_IMP.get, reverse=True)[:5]}')
else:
    print('No model found — skip attribution.')

---
## Phase I — Visualizations

In [ ]:
if RESULT and 'train_aucs' in RESULT:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Training curve
    ax = axes[0]
    ep = range(1, len(RESULT['train_aucs']) + 1)
    ax.plot(ep, RESULT['train_aucs'], color='#2196F3', lw=1.5, label='Train AUC')
    ax.plot(ep, RESULT['val_aucs'],   color='#FF6F00', lw=1.5, label='Val AUC')
    ax.axhline(RESULT['best_auc'], color='green', lw=1, ls='--',
               label=f'Best={RESULT["best_auc"]:.4f} @ ep {RESULT["best_epoch"]}')
    ax.set_title('Training Curve — Mamba v4', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
    ax.legend(); ax.grid(alpha=0.3)

    # Per-epoch timing
    ax = axes[1]
    ep_t = RESULT.get('ep_times', [])
    if ep_t:
        ax.plot(range(1, len(ep_t)+1), ep_t, color='#7B1FA2', lw=1.2)
        ax.axhline(np.mean(ep_t), color='gray', ls='--', lw=0.8,
                   label=f'avg={np.mean(ep_t):.1f}s')
        ax.set_title('Seconds per Epoch'); ax.set_xlabel('Epoch'); ax.set_ylabel('s')
        ax.legend(); ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'training_curves.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
if PERM_IMP:
    sorted_f = sorted(PERM_IMP, key=PERM_IMP.get, reverse=True)
    colors   = ['#2E7D32' if PERM_IMP[f] > 0 else '#C62828' for f in sorted_f]
    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(sorted_f) * 0.35)))

    # Permutation importance bar chart
    ax = axes[0]
    ax.barh(range(len(sorted_f)), [PERM_IMP[f] for f in sorted_f], color=colors)
    ax.set_yticks(range(len(sorted_f)))
    ax.set_yticklabels(sorted_f, fontsize=9)
    ax.invert_yaxis(); ax.set_xlabel('AUC Drop (higher = more important)')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('Permutation Importance — Mamba v4', fontweight='bold')

    # Boruta hit rates (from cache)
    ax = axes[1]
    if SELECTION_CACHE.exists():
        with open(SELECTION_CACHE) as f:
            sel_data = json.load(f)
        hit_rates = sel_data.get('hit_rates', {})
        if hit_rates:
            feat_hr = sorted(hit_rates.items(), key=lambda t: t[1], reverse=True)
            names_hr, rates_hr = zip(*feat_hr)
            colors_hr = ['#1565C0' if r > BORUTA_HIT_RATE else '#9E9E9E' for r in rates_hr]
            ax.barh(range(len(names_hr)), rates_hr, color=colors_hr)
            ax.set_yticks(range(len(names_hr)))
            ax.set_yticklabels(names_hr, fontsize=8)
            ax.invert_yaxis()
            ax.axvline(BORUTA_HIT_RATE, color='red', ls='--', lw=1,
                       label=f'Boruta threshold ({BORUTA_HIT_RATE})')
            ax.set_xlabel('Boruta hit rate'); ax.legend()
            ax.set_title('Boruta Hit Rates (all Stage 3 survivors)', fontweight='bold')

    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'feature_importance.png', dpi=130, bbox_inches='tight')
    plt.show()

---
## Phase J — Save Results

In [ ]:
out = {
    'notebook': '03_mamba_omni_0fee_v4', 'version': 'v4',
    'interval': INTERVAL, 'created': pd.Timestamp.now().isoformat(),
    'device': str(DEVICE),
    'architecture': {'d_model': D_MODEL, 'd_state': D_STATE, 'n_layers': N_LAYERS,
                     'scan_chunk': SCAN_CHUNK, 'batch': BATCH, 'seq_len': SEQ_LEN},
    'selection': {
        'label': 'label_tbm',
        'stages': ['variance+spearman', 'univariate_auc', 'rolling_auc_stability', 'boruta'],
        'boruta_trials': BORUTA_TRIALS,
        'selected_features': SELECTED_FEATURES,
    },
    'result': {k: v for k, v in RESULT.items() if k not in ('val_probs',)} if RESULT else {},
    'permutation_importance': PERM_IMP,
}
out_path = ARTS_DIR / 'results.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f'Saved: {out_path}')
if RESULT:
    print(f'\n── Summary ─────────────────────────────────────────────')
    print(f'  Features:    {RESULT["n_features"]} (Boruta-selected against TBM)')
    print(f'  Val AUC:     {RESULT["best_auc"]:.4f}  @ ep {RESULT["best_epoch"]}')
    print(f'  Train seqs:  {RESULT["n_train_seqs"]:,}')
    print(f'  Val seqs:    {RESULT["n_val_seqs"]:,}')
    print(f'  Train time:  {RESULT["elapsed_s"]/60:.1f} min')